In [ ]:
# Chapter 19: Training and Deploying TensorFlow Models at Scale

## Global Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os

# Check versions
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

TensorFlow Version: 2.19.0
Keras Version: 3.10.0


## Serving a TensorFlow Model
Once a model is trained, it needs to be deployed so that it can make predictions in a production environment. The simplest way is to save the model and load it in your application, but this doesn't scale well. TensorFlow Serving is a high-performance serving system designed for production environments. It can handle version management, batching, and model updates without downtime.

### Saving a Model for Serving
To serve a model, we first need to export it to the SavedModel format. This format is universal and contains the model architecture, weights, and the computation graph.

<p align="left"><img src="../fig/figure19.1.png" width="45%"></p>

### Code Example: Saving a Model

In [5]:
import tensorflow as tf
from tensorflow import keras
import os
import numpy as np

# Create a simple model for demonstration
model = keras.models.Sequential([
    keras.layers.Input(shape=[28, 28]),
    keras.layers.Flatten(),
    keras.layers.Dense(10, activation="softmax")
])

# Kita perlu melakukan build/compile atau satu kali forward pass
# agar model memiliki state yang pasti sebelum diexport
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.build(input_shape=(None, 28, 28))

model_version = "0001"
model_name = "my_mnist_model"
model_path = os.path.join(model_name, model_version)

# PERBAIKAN: Gunakan model.export() untuk Keras 3
# Ini akan menghasilkan format SavedModel (saved_model.pb + variables)
print(f"Exporting model to: {model_path}...")
model.export(model_path)

print(f"Model exported successfully.")
print("Directory contents:")
for root, dirs, files in os.walk(model_name):
    print(f"{root}/: {files}")

Exporting model to: my_mnist_model/0001...
Saved artifact at 'my_mnist_model/0001'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  136714573057872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136714573058640: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model exported successfully.
Directory contents:
my_mnist_model/: []
my_mnist_model/0001/: ['fingerprint.pb', 'saved_model.pb']
my_mnist_model/0001/variables/: ['variables.index', 'variables.data-00000-of-00001']
my_mnist_model/0001/assets/: []


Explanation: The saved_model.pb file defines the computation graph. The variables directory holds the weights. The version number (0001) allows TF Serving to automatically load the newest version.

### TensorFlow Serving


TF Serving can be installed via Docker (recommended) or APT. It exposes a REST API and a gRPC API.

- REST API: Easier to use, standard HTTP JSON requests.
- gRPC API: More efficient, uses Protocol Buffers, suitable for high-load internal communication.

To start TF Serving (Command Line / Docker):

docker run -it --rm -p 8500:8500 -p 8501:8501 \
  -v "$PWD/my_mnist_model:/models/my_mnist_model" \
  -e MODEL_NAME=my_mnist_model \
  tensorflow/serving

### Code Example: Querying TF Serving via REST API

In [7]:
import json
import requests

# Create dummy input data
X_new = np.random.rand(3, 28, 28).tolist()

# Prepare JSON request
request_json = json.dumps({
    "signature_name": "serving_default",
    "instances": X_new
})

# In a real scenario, this URL would be localhost or a server IP
# response = requests.post("http://localhost:8501/v1/models/my_mnist_model:predict", data=request_json)
# predictions = response.json()["predictions"]

# Simulated Response
print("Sending request to http://localhost:8501/v1/models/my_mnist_model:predict...")
print("Received response with shape (3, 10)") # 3 images, 10 classes
print("Prediction for first image (class probabilities):")
# Simulate output
print("[0.1, 0.05, 0.05, 0.7, 0.0, 0.05, 0.0, 0.05, 0.0, 0.0]")

Sending request to http://localhost:8501/v1/models/my_mnist_model:predict...
Received response with shape (3, 10)
Prediction for first image (class probabilities):
[0.1, 0.05, 0.05, 0.7, 0.0, 0.05, 0.0, 0.05, 0.0, 0.0]


## Deploying to Mobile and Embedded Devices (TensorFlow Lite)
TensorFlow Lite (TFLite) is a lightweight library specifically designed for mobile devices (Android/iOS) and embedded systems (Raspberry Pi, Microcontrollers). TFLite models are compressed and optimized for low latency and small binary size.

<p align="left"><img src="../fig/figure19.8.png" width="45%"></p>

### The Converter
To use TFLite, we must convert the standard TensorFlow model into a .tflite file using the TFLiteConverter.

### Optimizations:

- Quantization: Converts 32-bit floats (weights) to 16-bit floats or 8-bit integers. This dramatically reduces model size (up to 4x) and speeds up inference, with often negligible loss in accuracy.

### Code Example: Converting and Quantizing a Model

In [8]:
converter = tf.lite.TFLiteConverter.from_saved_model(model_path)

# Normal Conversion
tflite_model = converter.convert()

# Optimization (Quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

# Save the files
with open("model.tflite", "wb") as f:
    f.write(tflite_model)
with open("model_quantized.tflite", "wb") as f:
    f.write(tflite_quantized_model)

print(f"Original Model Size: {len(tflite_model) / 1024:.2f} KB")
print(f"Quantized Model Size: {len(tflite_quantized_model) / 1024:.2f} KB")

Original Model Size: 32.05 KB
Quantized Model Size: 9.24 KB


Explanation: The quantized model is significantly smaller. This is crucial for app download sizes and memory usage on edge devices.

## Running on the Browser (TensorFlow.js)
TensorFlow.js (TF.js) allows us to run ML models directly in a web browser using JavaScript.
- Benefits: Zero latency (no server round-trip), data privacy (data stays on the user's device), and access to device sensors (camera, mic) via the browser.
- Usage: You can convert an existing Python model to TF.js format or train a model directly in the browser.

To convert a saved model:

In [ ]:
tensorflowjs_converter --input_format=tf_saved_model \
    ./my_mnist_model/0001 \
    ./my_tfjs_model

This produces a model.json (graph structure) and binary shard files (weights).

## Distributed Training at Scale
When models are too large to fit in one GPU, or training takes too long, we use Distributed Training.

### Strategies
1. Data Parallelism: The model is replicated across every device (GPU/TPU). Each device processes a different batch of data. Gradients are computed locally and then aggregated (averaged) across all devices before updating the weights.
- MirroredStrategy: Supports synchronous distributed training on multiple GPUs on a single machine. It keeps variables in sync by mirroring them across devices.
- MultiWorkerMirroredStrategy: Similar to MirroredStrategy, but for multiple GPUs across multiple machines.
2. Model Parallelism: The model is split across different devices. One part of the model runs on GPU 0, the next part on GPU 1, etc. This is harder to implement and usually requires pipelining to keep all GPUs busy.

### Code Example: Using MirroredStrategy (Data Parallelism)
Using tf.distribute.MirroredStrategy is incredibly easy. You simply wrap the model creation and compilation inside the strategy's scope.

In [13]:
try:
    # Detect GPUs, fallback to CPU if none
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        strategy = tf.distribute.MirroredStrategy()
        print(f"Running on {len(gpus)} GPUs")
    else:
        strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
        print("Running on CPU (OneDeviceStrategy)")

    # Wrap model creation in the strategy scope
    with strategy.scope():
        model_dist = keras.models.Sequential([
            keras.layers.Input(shape=[28, 28]),
            keras.layers.Flatten(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dense(10, activation="softmax")
        ])
        model_dist.compile(loss="sparse_categorical_crossentropy",
                           optimizer="adam", metrics=["accuracy"])

    # Load data (simulated)
    X_train = np.random.rand(100, 28, 28).astype(np.float32)
    y_train = np.random.randint(0, 10, size=(100,))

    # Train normally
    print("Starting distributed training...")
    history = model_dist.fit(X_train, y_train, epochs=20, batch_size=32, verbose=2)

except RuntimeError as e:
    print(e)

Running on CPU (OneDeviceStrategy)
Starting distributed training...
Epoch 1/20
4/4 - 1s - 259ms/step - accuracy: 0.1800 - loss: 2.3434
Epoch 2/20
4/4 - 0s - 44ms/step - accuracy: 0.2600 - loss: 2.1314
Epoch 3/20
4/4 - 0s - 43ms/step - accuracy: 0.2900 - loss: 1.9301
Epoch 4/20
4/4 - 0s - 46ms/step - accuracy: 0.4000 - loss: 1.8139
Epoch 5/20
4/4 - 0s - 42ms/step - accuracy: 0.5100 - loss: 1.6818
Epoch 6/20
4/4 - 0s - 41ms/step - accuracy: 0.2800 - loss: 1.6998
Epoch 7/20
4/4 - 0s - 41ms/step - accuracy: 0.5800 - loss: 1.5113
Epoch 8/20
4/4 - 0s - 43ms/step - accuracy: 0.6600 - loss: 1.4205
Epoch 9/20
4/4 - 0s - 42ms/step - accuracy: 0.6500 - loss: 1.3236
Epoch 10/20
4/4 - 0s - 47ms/step - accuracy: 0.6700 - loss: 1.2383
Epoch 11/20
4/4 - 0s - 42ms/step - accuracy: 0.8600 - loss: 1.1455
Epoch 12/20
4/4 - 0s - 42ms/step - accuracy: 0.9000 - loss: 1.0175
Epoch 13/20
4/4 - 0s - 41ms/step - accuracy: 0.9100 - loss: 0.9557
Epoch 14/20
4/4 - 0s - 43ms/step - accuracy: 0.9300 - loss: 0.9045
Ep

Explanation: If multiple GPUs were available, MirroredStrategy would have placed a replica of the model on each one. The batch_size is usually the global batch size (split automatically among replicas). The code remains almost identical to non-distributed code.

### Training on TPUs (Tensor Processing Units)
TPUs are custom AI accelerators developed by Google. To use them (e.g., on Colab), you use TPUStrategy. The only difference is the initialization logic to connect to the TPU cluster.

In [14]:
# Pseudo-code for TPU initialization
# resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
# tf.config.experimental_connect_to_cluster(resolver)
# tf.tpu.experimental.initialize_tpu_system(resolver)
# strategy = tf.distribute.TPUStrategy(resolver)
# with strategy.scope(): ...

This concludes the comprehensive summary of Chapter 19, covering the entire lifecycle from saving models for production to deploying them on edge devices and scaling training across massive clusters.